In [2]:
#### save into json
import json
def save(path, data):    
    # Save the dictionary to a JSON file
    with open(path, 'w') as json_file:
        json.dump(data, json_file)

#### load from json
import json
def load(path):
    # Load the JSON file
    with open(path, 'r') as json_file:
        return json.load(json_file)

all_detections_path = '/home/jupyter/test/PuTR/output/Copy of A iORA Isetan CHANNEL  3  (1100-2330)      1 MAR 2025_segment_1.json'
all_detections = load(all_detections_path)
#detections = all_detections['frames'] #src
detections = all_detections

In [3]:
#video_path = "/home/jupyter/test/muggled_sam/inputs/uid_vid_00000.mp4"
video_path = '/home/jupyter/test/PuTR/output/Copy of A iORA Isetan CHANNEL  3  (1100-2330)      1 MAR 2025_segment_1.mp4'
#output_path = "/home/jupyter/test/muggled_sam/outputs/uid_vid_00000.mp4"
output_path = "/home/jupyter/test/muggled_sam/outputs/SKIP_Copy of A iORA Isetan CHANNEL  3  (1100-2330)      1 MAR 2025_segment_1.mp4"

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# This is a hack to make this script work from outside the root project folder (without requiring install)
try:
    import lib  # NOQA
except ModuleNotFoundError:
    import os
    import sys

    #parent_folder = os.path.dirname(os.path.dirname(__file__))
    parent_folder = os.path.dirname(os.path.dirname(os.path.abspath('/home/jupyter/test/muggled_sam/simple_examples/iora.ipynb')))
    if "lib" in os.listdir(parent_folder):
        sys.path.insert(0, parent_folder)
    else:
        raise ImportError("Can't find path to lib folder!")

from collections import defaultdict
import cv2
import numpy as np
import torch
from lib.v2_sam.make_sam_v2 import make_samv2_from_original_state_dict
from lib.demo_helpers.video_data_storage import SAM2VideoObjectResults, SimpleSamurai
import supervision as sv
import time
from tqdm import tqdm
from skimage.morphology import remove_small_objects



# Define pathing & device usage
model_path = "/home/jupyter/test/muggled_sam/model_weights/sam2.1_hiera_base_plus.pt"
device, dtype = "cpu", torch.float32
if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16

# Define image processing config (shared for all video frames)
imgenc_config_dict = {"max_side_length": 1024, "use_square_sizing": True}

# Set up memory storage for tracked objects
# -> Assumes each object is represented by a unique dictionary key (e.g. 'obj1')
# -> This holds both the 'prompt' & 'recent' memory data needed for tracking!
memory_per_obj_dict = defaultdict(SAM2VideoObjectResults.create)

# Read first frame to check that we can read from the video, then reset playback
vcap = cv2.VideoCapture(video_path)
vcap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)  # See: https://github.com/opencv/opencv/issues/26795
ok_frame, first_frame = vcap.read()
if not ok_frame:
    raise IOError(f"Unable to read video frames: {video_path}")
vcap.set(cv2.CAP_PROP_POS_FRAMES, 0)
total_frames = int(vcap.get(cv2.CAP_PROP_FRAME_COUNT))  # Get total frame count for progress bar
total_use_frames = 0
# Iterate through each item (key, value pair) in the dictionary
for key, value in detections.items():
    # Get the 'scores' list, default to an empty list if the key doesn't exist
    scores = value.get('scores', [])
    # Get the 'boxes' list, default to an empty list if the key doesn't exist
    boxes = value.get('boxes', [])

    # Check the conditions:
    # 1. The 'scores' list is not empty (lists are truthy if not empty)
    # 2. At least one score in the 'scores' list is greater than 0 (using the any() function)
    # 3. The 'boxes' list is not empty
    if scores and any(score > 0 for score in scores) and boxes:
        total_use_frames += 1

# Print the final count
print('number of frames with none-zero detections:', total_use_frames)

# Set up model
print("Loading model...")
model_config_dict, sammodel = make_samv2_from_original_state_dict(model_path)
sammodel.to(device=device, dtype=dtype)
#from rfdetr import RFDETRBase
#model = RFDETRBase()

# How often to redo detection
step = 15 # no use
skip_frames = 5 # no use
# How many tracking frames it can be occluded before it gets removed
# E.g threshold of 10 at skip_frames of 5 => 10/(15/5) 3.33s before its removed after complete occlusion
# Increasing this threshold increases processing time
occlusion_threshold = 10
# At what SAM2 object score to remove for display to reduce artifacts
obj_score_threshold = 2.5
# IOU Threshold
iou_threshold = 0.5
# object tracker index
object_index = 0
# tqdm progress bar
progress_bar = None

mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.TRACK)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK)

def get_existing_mask(frame, mask):
    non_none_masks = [object_memory.mask for _, object_memory in memory_per_obj_dict.items() if object_memory.mask is not None]

    if len(non_none_masks) > 0:
        obj_mask = torch.nn.functional.interpolate(
            mask,
            size=frame.shape[:2],
            mode="bilinear",
            align_corners=False,
        )
        obj_mask_binary = (obj_mask > 0.0).cpu().numpy().squeeze()

        iou_scores = sv.mask_iou_batch(np.array(non_none_masks), np.array([obj_mask_binary]))

        if np.any(iou_scores > iou_threshold):
            return True
    
    return None

global output_dict
output_dict = {}
global count
count = 0
def callback(frame, frame_idx):
    #print('frame_idx', frame_idx)    
        
    global stored_annotated_frame    
    if int(frame_idx) == 0:
        stored_annotated_frame = frame
        return frame
    
    if (str(frame_idx) not in detections):
        return None
    
    detection = detections[str(frame_idx)]
    boxes = detection['boxes']
    scores = detection['scores']
    
    #print('boxes', boxes)
    #print('scores', scores)
    #det_labels = np.ones(len(detection['scores']), dtype=int)
                
    #if det_labels.size == 0:
    if scores and any(score > 0 for score in scores) and boxes:
        pass
    else:
        #print('empty detections')
        return None
    
    # tqdm progress bar
    global progress_bar
    if progress_bar is None:
        progress_bar = tqdm(total=total_use_frames, desc="Processing Video")
    progress_bar.update(1)

    if frame_idx == 0 or True:
    #if (frame_idx % skip_frames == 0):
        # Encode frame data (shared for all objects)
        encoded_imgs_list, _, _ = sammodel.encode_image(frame, **imgenc_config_dict)

        # Storage for results
        label_result = []
        mask_result = []
        
        marked_for_deletion = []
        # Update tracking using newest frame
        for obj_key_name, obj_memory in memory_per_obj_dict.items():
            # obj_score, best_mask_idx, mask_preds, mem_enc, obj_ptr = sammodel.step_video_masking(
            #         encoded_imgs_list, **obj_memory.to_dict()
            #     )

            # obj_score = obj_score.item()
            # if obj_score < 0:
            #     obj_memory.increment_bad_ctr()
            #     if obj_memory.bad_ctr > occlusion_threshold:
            #         marked_for_deletion.append(obj_key_name)
            #     continue

            # Samurai Implementation
            obj_score, is_mem_ok, best_mask_pred, mem_enc, obj_ptr, xy1xy2_kal = obj_memory.samurai.step_video_masking(
                sammodel, encoded_imgs_list, **obj_memory.to_dict()
            )

            # Store memory if samurai says its ok for tracking
            if is_mem_ok:
                # Store 'recent' memory encodings from current frame (helps track objects with changing appearance)
                # -> This can be commented out and tracking may still work, if object doesn't change much
                obj_memory.store_result(frame_idx, mem_enc, obj_ptr)
            else:
                obj_memory.increment_bad_ctr()
                if obj_memory.bad_ctr > occlusion_threshold:
                    marked_for_deletion.append(obj_key_name)
                    continue

            # Add object mask prediction to 'combine' mask for display
            # -> This is just for visualization, not needed for tracking
            obj_mask = torch.nn.functional.interpolate(
                best_mask_pred,
                size=frame.shape[:2],
                mode="bilinear",
                align_corners=False,
            )
            obj_mask_binary = (obj_mask > 0.0).cpu().numpy().squeeze()

            obj_memory.store_mask(obj_mask_binary)

            # But skip it for display if score is bad to reduce artifacts.
            if obj_score < obj_score_threshold:
                continue
            
            obj_mask_binary = remove_small_objects(obj_mask_binary, 128)
            label_result.append(int(obj_key_name))
            mask_result.append(obj_mask_binary)
            
        for to_delete in marked_for_deletion:
            # print(to_delete)
            memory_per_obj_dict.pop(to_delete, None)

        # Doing detection after the mask are updated for this frame improves results
        frame_prompts_dict = {}
        #if (frame_idx == 0 or True) and (str(frame_idx) in detections):
        if True:
            global count
            count += 1
        #if (frame_idx % step == 0):
            # Generate prompt dict from detection model
            #detections = model.predict(frame, threshold=0.35)
            #detections = detections[detections.class_id == 1]
            #print('rfdetr: ', detections)
            #detection = detections[str(frame_idx)]
            #print('own: ', detection)
            det_bboxes = np.array(detection['boxes'])#.astype(np.float32)
            #print('det_bboxes', det_bboxes)
            image_height, image_width = frame.shape[:2]
            for i, box in enumerate(det_bboxes):
            #for i, box in enumerate(detections.xyxy):
                x1, y1, x2, y2 = box
                norm_x1 = x1 / image_width
                norm_y1 = y1 / image_height
                norm_x2 = x2 / image_width
                norm_y2 = y2 / image_height

                # Clamp values to [0.0, 1.0] to handle potential rounding errors
                # or boxes slightly outside the frame.
                norm_x1 = max(0.0, min(1.0, norm_x1))
                norm_y1 = max(0.0, min(1.0, norm_y1))
                norm_x2 = max(0.0, min(1.0, norm_x2))
                norm_y2 = max(0.0, min(1.0, norm_y2))
                formatted_box_list = [(float(norm_x1), float(norm_y1)), (float(norm_x2), float(norm_y2))]

                frame_prompts_dict[i] = {
                    "box_tlbr_norm_list": [formatted_box_list],
                    "fg_xy_norm_list": [],
                    "bg_xy_norm_list": [],
                }
        else:
            return None
                
        
        # Moved prompt generation to after mask generation for better iou as you are not comparing against the previous frame
        # Generate & store prompt memory encodings for each object as needed
        prompts_dict = frame_prompts_dict if frame_prompts_dict else None
        if prompts_dict is not None:

            # Loop over all sets of prompts for the current frame
            for obj_key_name, obj_prompts in prompts_dict.items():
                # print(f"Generating prompt for object: {obj_key_name} (frame {frame_idx})")
                init_mask, init_mem, init_ptr = sammodel.initialize_video_masking(encoded_imgs_list, **obj_prompts)
                    
                existing_masks = get_existing_mask(frame, init_mask)

                if existing_masks is None:
                    global object_index
                    object_index = (object_index or 0) + 1
                    memory_per_obj_dict[object_index].store_prompt_result(frame_idx, init_mem, init_ptr)
                    samurai = SimpleSamurai(init_mask)
                    memory_per_obj_dict[object_index].store_samurai(samurai)

                # Directly show mask for first frame
                if frame_idx == 0:
                    obj_mask = torch.nn.functional.interpolate(
                        init_mask,
                        size=frame.shape[:2],
                        mode="bilinear",
                        align_corners=False,
                    )
                    obj_mask_binary = (obj_mask > 0.0).cpu().numpy().squeeze()
                    obj_mask_binary = remove_small_objects(obj_mask_binary)
                    label_result.append(int(object_index))
                    mask_result.append(obj_mask_binary)
       
        global output_dict
        trk_ids = np.array(label_result)
        dets = sv.mask_to_xyxy(np.array(mask_result))
        output_dict[frame_idx] = {trk_id: det.tolist() for trk_id, det in zip(trk_ids, dets)}
       
        #global stored_annotated_frame
        
        if len(mask_result) == 0:
            stored_annotated_frame = frame
            return frame

        # Write results to frame
        mask_detections = sv.Detections(
            xyxy=sv.mask_to_xyxy(np.array(mask_result)),
            mask=np.array(mask_result),
            tracker_id=np.array(label_result)
        )
        annotated_frame = mask_annotator.annotate(
            scene=frame.copy(),
            detections=mask_detections,
        )
        annotated_frame = box_annotator.annotate(
            scene=annotated_frame,
            detections=mask_detections,
        )
        annotated_frame = label_annotator.annotate(
            scene=annotated_frame,
            detections=mask_detections,
            labels=[str(i) for i in label_result]
        )

        stored_annotated_frame = annotated_frame

    return stored_annotated_frame

process_time_start = time.time()

try:
    # Your main processing logic that might be interrupted
    sv.process_video(
        source_path=video_path,
        target_path=output_path,
        callback=callback
    )
except KeyboardInterrupt:
    # This block will be executed if you press Ctrl+C
    print("\nProcessing interrupted by user.")
    # You might want to save partial results here if applicable
finally:
    # This block will ALWAYS be executed, whether the 'try' block
    # completes successfully, encounters another error, or is interrupted.
    if progress_bar:
        progress_bar.close()
    print("Progress bar closed.") # Optional: confirmation message

process_time = time.time() - process_time_start 
print('process_time', process_time)

number of frames with none-zero detections: 6198
Loading model...


Processing Video: 100%|█████████▉| 6197/6198 [1:58:57<00:01,  1.15s/it]  

Progress bar closed.
process_time 7137.84152007103


In [5]:
print('process_time', process_time)
print('no frame with tracking and none-zero detection', count)
print('time/frame', process_time / count)
print('frame/second', count / process_time)

process_time 7137.84152007103
no frame with tracking and none-zero detection 6197
time/frame 1.1518220945733468
frame/second 0.8681896316378754


In [6]:
# Define the path to your output file
output_file_path = "/home/jupyter/test/muggled_sam/outputs/my_results.txt"

# Open the file in write mode ('w') within a 'with' statement
# 'w' mode creates the file if it doesn't exist, and overwrites it if it does.
with open(output_file_path, 'w') as f:
    # Use the .write() method to write the string to the file
    result_string = f"Process time: {process_time}\n"
    result_string += f"Number of frames with tracking: {count}\n"
    result_string += f"Time per frame: {process_time / count}\n"
    result_string += f"Frames per second: {count / process_time}\n"
    # Write the string to the file
    f.write(result_string)

print(f"Successfully wrote results to {output_file_path}")

Successfully wrote results to /home/jupyter/test/muggled_sam/outputs/my_results.txt


In [7]:
def convert_output(track_list):
    # Dictionary to store the transformed data
    output_data = {}

    # Iterate through each frame and its tracking data
    for frame, track_data_in_frame in track_list.items():
        # Iterate through each tracked object (ID and box) in the current frame
        for object_id, bounding_box in track_data_in_frame.items():
            # Check if this object_id is already in our output structure
            
            if str(object_id) not in output_data:
                # If not, initialize its entry with empty lists for frame and box
                object_id_str = str(object_id)
                output_data[object_id_str] = {"frames": [], "boxes": []}

            # Append the current frame number and bounding box to the respective lists
            # for this object ID
            object_id_str = str(object_id)
            output_data[object_id_str]["frames"].append(frame)
            output_data[object_id_str]["boxes"].append(bounding_box)

    return output_data

output = convert_output(output_dict)
output_tracking_path = os.path.join('/home/jupyter/test/muggled_sam/outputs/', 'output_tracking.json')
save(output_tracking_path, output)